# Preparing Data for the Algorithm

## 1. Load all data

In [1]:
# Manejo de datos y operaciones matemáticas
import pandas as pd
import numpy as np

# Visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid") # Configuración estética de Seaborn

# Procesamiento de imágenes/vídeo
import cv2

# Lectura de metadatos y archivos serializados
import json
import pickle
import os, sys # Útil para manejar las rutas 

# Para cargar datos
from kloppy import sportscode, metrica
sys.path.append('..')
import games

In [2]:
# Load Tagging
sys.path.insert(0, os.path.abspath('..'))
print("Cargando Smart Tagging (Eventos)...")
path = games.asset_paths()['smart_tagging']
events_df = sportscode.load(path).to_df()
print(f"-> {len(events_df)} instancias cargadas con éxito.")
display(events_df.head(20))

Cargando Smart Tagging (Eventos)...
-> 638 instancias cargadas con éxito.


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,None,None
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,None,None
5,6,1,0 days 01:48:40,0 days 01:48:45,SET PIECES,N/A,Kick Off (After Goal),2nd Half,None,None,None
6,7,1,0 days 01:17:41.720000,0 days 01:18:09.400000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,4,None,None
7,8,1,0 days 01:18:42.760000,0 days 01:19:07.040000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,4,None,None
8,9,1,0 days 01:21:23.200000,0 days 01:21:44.600000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,2,None,None
9,10,1,0 days 01:22:46.080000,0 days 01:23:11.160000,PLAYERS IN THE BOX,Barcelona,None,2nd Half,2,None,None


In [4]:
# Load ATD
sys.path.insert(0, os.path.abspath('..'))

print("Cargando Tracking ATD...")
paths = games.asset_paths(source='tactical')

dataset = metrica.load_tracking_epts(
    meta_data=paths['atd_meta'],
    raw_data=paths['atd_raw'], 
)

print(f"-> {len(dataset.records)} frames cargados con éxito.")
frame_df = dataset.to_df()
display(frame_df.head())

Cargando Tracking ATD...
-> 199319 frames cargados con éxito.


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,44_d,44_s,45_x,45_y,45_d,45_s,46_x,46_y,46_d,46_s
0,1,0 days 00:00:00.040000,1,None,None,NaN,NaN,None,None,0.33411,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
1,1,0 days 00:00:00.080000,2,None,None,NaN,NaN,None,None,0.33404,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
2,1,0 days 00:00:00.120000,3,None,None,NaN,NaN,None,None,0.33396,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
3,1,0 days 00:00:00.160000,4,None,None,NaN,NaN,None,None,0.33389,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
4,1,0 days 00:00:00.200000,5,None,None,NaN,NaN,None,None,0.33382,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN


## 2. Structure Dataset

In [5]:
import pandas as pd
import numpy as np

def build_hackathon_dataset_by_code(tracking_df, events_df, fps=25):
    # 1. Ordenar el tracking cronológicamente por frame_id
    df = tracking_df.sort_values('frame_id').reset_index(drop=True)
    
    # 2. Asegurar cálculo de frames de inicio y fin desde timedelta
    if 'start_frame' not in events_df.columns:
        if pd.api.types.is_timedelta64_dtype(events_df['timestamp']):
            start_seconds = events_df['timestamp'].dt.total_seconds()
        else:
            start_seconds = pd.to_numeric(events_df['timestamp'], errors='coerce')
        events_df['start_frame'] = (start_seconds * fps).astype(int)
        
    if 'end_frame' not in events_df.columns:
        if pd.api.types.is_timedelta64_dtype(events_df['end_timestamp']):
            end_seconds = events_df['end_timestamp'].dt.total_seconds()
        else:
            end_seconds = pd.to_numeric(events_df['end_timestamp'], errors='coerce')
        events_df['end_frame'] = (end_seconds * fps).astype(int)
        
    # --- DIAGNÓSTICO RÁPIDO PARA VER SI COINCIDEN ---
    print(f"Rango de frame_id en tracking: {df['frame_id'].min()} a {df['frame_id'].max()}")
    print(f"Rango de start_frame en events: {events_df['start_frame'].min()} a {events_df['start_frame'].max()}")
    # -----------------------------------------------

    unique_codes = events_df['code'].dropna().unique()
    print(unique_codes )
    
    for code in unique_codes:
        clean_code = str(code).strip().lower().replace(' ', '_').replace('-', '_')
        
        col_value = f"{clean_code}"
        col_team = f"{clean_code}_team"
        
        # IMPORTANTE: Inicializamos como object/None para aceptar texto sin warnings
        df[col_value] = None
        df[col_team] = None
        
        sub_events = events_df[events_df['code'] == code]
        
        for _, ev in sub_events.iterrows():
            start = ev['start_frame']
            end = ev['end_frame']
            
            # Condicional para set pieces vs el resto
            if 'PLAYERS IN THE BOX' in code:
                val = ev.get('Max Players in the box', None)
            elif 'SET PIECES' in code:
                val = ev.get('Type', None)
            else:
                val = True  # Indicador booleano de presencia para el resto de eventos
                
            team = ev.get('Team', None)
            
            # Máscara de los frames
            mask = (df['frame_id'] >= start) & (df['frame_id'] <= end)
            
            df.loc[mask, col_value] = val
            df.loc[mask, col_team] = team
            
    return df

In [6]:
# Ejecutamos la función (asumiendo que el framerate de tu vídeo es de 25 fps, cámbialo si es 50fps)
df_hackathon = build_hackathon_dataset_by_code(frame_df, events_df, fps=25)

Rango de frame_id en tracking: 1 a 199319
Rango de start_frame en events: 11069 a 191965
['SET PIECES' 'PLAYERS IN THE BOX' 'BALL IN FINAL THIRD' 'BALL IN THE BOX'
 'GOALS' 'PROGRESSION' 'ATTACKING TRANSITION' 'BUILD UP' 'LONG BALL'
 'CREATING CHANCES' 'DEFENDING IN MIDDLE THIRD' 'DEFENSIVE TRANSITION'
 'DEFENDING IN ATTACKING THIRD' 'DEFENDING IN DEFENSIVE THIRD']


In [7]:
# 1. Definimos las columnas específicas que quieres conservar además de las que terminan en '_d'
cols_especificas = [
    'ball_state', 
    'ball_owning_team_id', 
    'ball_z', 
    'set_pieces_team'
]

# 2. Buscamos todas las columnas del DataFrame que terminan en '_d'
cols_terminan_en_d = [col for col in df_hackathon.columns if col.endswith('_d')]

# 3. Unimos ambas listas asegurándonos de que no haya duplicados (usando set) y mantenemos el orden si es necesario
# O directamente filtramos el DataFrame combinándolas de forma segura:
cols_a_borrar = list(set(cols_terminan_en_d + [c for c in cols_especificas if c in df_hackathon.columns]))

# 4. Creamos el nuevo DataFrame filtrado
df_hackathon = df_hackathon.drop(columns=cols_a_borrar)

In [8]:
def ball_speed(
    current_ball,
    previous_ball,
    fps=25
):
    if current_ball is None:
        return None

    if previous_ball is None:
        return None

    distance = current_ball - previous_ball

    return distance * fps

In [9]:
cols_terminan_en_x = [col for col in df_hackathon.columns if col.endswith('_x')]
cols_terminan_en_y = [col for col in df_hackathon.columns if col.endswith('_y')]

for col in cols_terminan_en_x:
    speeds = []
    current_balls = df_hackathon[col]
    previous_balls = df_hackathon[col].shift(1)
    col_name= f"{col}_vel"

    for curr, prev in zip(current_balls, previous_balls):
    # Comprobamos si alguno es NaN antes de pasar a numpy arrays
        if pd.isna(curr) or pd.isna(prev):
            speeds.append(None)
        else:
            # Llamamos a tu función exacta convirtiendo a numpy arrays
            speed = ball_speed(
                np.array(curr), 
                np.array(prev), 
                fps=25
            )
            speeds.append(speed)

    df_hackathon[col_name] = speeds
     

for col in cols_terminan_en_y:
    speeds = []
    current_balls = df_hackathon[col]
    previous_balls = df_hackathon[col].shift(1)
    col_name= f"{col}_vel"

    for curr, prev in zip(current_balls, previous_balls):
    # Comprobamos si alguno es NaN antes de pasar a numpy arrays
        if pd.isna(curr) or pd.isna(prev):
            speeds.append(None)
        else:
            # Llamamos a tu función exacta convirtiendo a numpy arrays
            speed = ball_speed(
                np.array(curr), 
                np.array(prev), 
                fps=25
            )
            speeds.append(speed)

    df_hackathon[col_name] = speeds

display(df_hackathon[['frame_id', col, col_name]].head(52000))   

C:\Users\Maria Fité\AppData\Local\Temp\ipykernel_9760\2018402363.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_hackathon[col_name] = speeds
C:\Users\Maria Fité\AppData\Local\Temp\ipykernel_9760\2018402363.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_hackathon[col_name] = speeds
C:\Users\Maria Fité\AppData\Local\Temp\ipykernel_9760\2018402363.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joinin

,frame_id,46_y,46_y_vel
0,1,0.39965,NaN
1,2,0.39965,0.00000
2,3,0.39965,0.00000
3,4,0.39965,0.00000
4,5,0.39965,0.00000
...,...,...,...
51995,51996,0.40661,0.01900
51996,51997,0.40736,0.01875
51997,51998,0.40808,0.01800
51998,51999,0.40879,0.01775


In [10]:
def ball_speed(current_ball, previous_ball, fps=25):

    if current_ball is None:
        return None

    if previous_ball is None:
        return None

    distance = np.linalg.norm(current_ball -previous_ball)

    return distance * fps

In [11]:
import numpy as np

# 1. Define las columnas exactas donde tienes las coordenadas de la pelota 
# (cambia 'ball_x' y 'ball_y' por los nombres que use tu tracking si son diferentes, ej: ball_x_m)
x_col = 'ball_x'
y_col = 'ball_y'

# 2. Agrupamos las coordenadas actuales en un array de tuplas o listas por fila
current_balls = list(zip(df_hackathon[x_col], df_hackathon[y_col]))

# 3. Obtenemos las coordenadas anteriores desplazando una posición hacia abajo (.shift(1))
previous_x = df_hackathon[x_col].shift(1)
previous_y = df_hackathon[y_col].shift(1)
previous_balls = list(zip(previous_x, previous_y))

# 4. Aplicamos tu función iterando de forma eficiente con zip
# (El primer frame se quedará en None porque no tiene anterior, tal como contempla tu función)
speeds = []
for curr, prev in zip(current_balls, previous_balls):
    # Comprobamos si alguno es NaN antes de pasar a numpy arrays
    if pd.isna(curr[0]) or pd.isna(curr[1]) or pd.isna(prev[0]) or pd.isna(prev[1]):
        speeds.append(None)
    else:
        # Llamamos a tu función exacta convirtiendo a numpy arrays
        speed = ball_speed(
            np.array(curr), 
            np.array(prev), 
            fps=25
        )
        speeds.append(speed)

# 5. Asignamos el resultado a la nueva columna del DataFrame
df_hackathon['ball_speed'] = speeds

print("¡Columna 'ball_speed' aplicada con éxito usando tu función!")
display(df_hackathon[['frame_id', x_col, y_col, 'ball_speed']].head(52000))

¡Columna 'ball_speed' aplicada con éxito usando tu función!


,frame_id,ball_x,ball_y,ball_speed
0,1,NaN,NaN,NaN
1,2,NaN,NaN,NaN
2,3,NaN,NaN,NaN
3,4,NaN,NaN,NaN
4,5,NaN,NaN,NaN
...,...,...,...,...
51995,51996,0.27556,0.56000,0.191159
51996,51997,0.27508,0.56744,0.186387
51997,51998,0.27463,0.57469,0.181599
51998,51999,0.27421,0.58173,0.176313


## 3. View Dataset Caractheristics

In [14]:
import pandas as pd

# 1. Ampliamos los límites de visualización para que no oculte filas de la tabla
pd.set_option('display.max_rows', None)

# 2. Volvemos a definir la función (o úsala tal cual la tienes)
def inspect_hackathon_dataset(df):
    print("=== 1. DIMENSIONES DEL DATASET ===" )
    print(f"Total de frames (filas): {df.shape[0]:,}")
    print(f"Total de columnas: {df.shape[1]}")
    
    print("\n=== 2. TIPOS DE DATOS Y VALORES NULOS ===")
    info_df = pd.DataFrame({
        'Tipo de Dato': df.dtypes,
        'Nulos': df.isnull().sum(),
        '% Nulos': (df.isnull().mean() * 100).round(2)
    })
    # Ahora mostrará toda la lista completa de columnas sin truncar el medio
    display(info_df)
    
    print("\n=== 3. RESUMEN ESTADÍSTICO (VARIABLES NUMÉRICAS) ===")
    display(df.describe())
    
    print("\n=== 4. ESTADÍSTICAS DE VARIABLES CATEGÓRICAS Y EVENTOS ===")
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    display(df[cat_cols].describe())

# 3. Ejecútalo de nuevo
inspect_hackathon_dataset(df_hackathon)

=== 1. DIMENSIONES DEL DATASET ===
Total de frames (filas): 199,319
Total de columnas: 265

=== 2. TIPOS DE DATOS Y VALORES NULOS ===


,Tipo de Dato,Nulos,% Nulos
period_id,int64,0,0.00
timestamp,timedelta64[ns],0,0.00
frame_id,int64,0,0.00
ball_x,float64,85831,43.06
ball_y,float64,85831,43.06
ball_speed,float64,86016,43.15
1_x,float64,119645,60.03
1_y,float64,119645,60.03
1_s,float64,123338,61.88
2_x,float64,121211,60.81



=== 3. RESUMEN ESTADÍSTICO (VARIABLES NUMÉRICAS) ===


,period_id,timestamp,frame_id,ball_x,ball_y,ball_speed,1_x,1_y,1_s,2_x,...,36_y_vel,37_y_vel,38_y_vel,39_y_vel,40_y_vel,41_y_vel,42_y_vel,43_y_vel,45_y_vel,46_y_vel
count,199319.0,199319,199319.00000,113488.000000,113488.000000,113303.000000,79674.000000,79674.000000,75981.000000,78108.000000,...,82794.000000,118622.000000,52415.000000,38044.000000,73488.000000,111051.000000,60313.000000,87216.000000,121226.00000,178906.000000
mean,1.0,0 days 01:06:26.400000,99660.00000,0.502648,0.570560,0.101237,0.473240,0.536663,1.875471,0.495088,...,0.000456,0.000227,0.000379,0.000739,-0.000340,0.000273,0.000054,-0.000772,-0.00009,0.000007
std,0.0,0 days 00:38:21.543339587,57538.58349,0.239323,0.284386,0.115901,0.193351,0.198242,1.507069,0.211607,...,0.021662,0.022355,0.022183,0.021775,0.024128,0.022836,0.019898,0.021236,0.02370,0.009376
min,1.0,0 days 00:00:00.040000,1.00000,-0.065170,-0.107130,0.000000,0.002240,0.007940,0.002636,0.003060,...,-0.153750,-0.227500,-0.263250,-0.100750,-0.444250,-0.162250,-0.097500,-0.133750,-0.16225,-0.153000
25%,1.0,0 days 00:33:13.220000,49830.50000,0.318747,0.356928,0.030192,0.337990,0.406223,0.890337,0.360470,...,-0.009750,-0.010750,-0.008750,-0.010000,-0.011000,-0.011000,-0.010250,-0.010750,-0.01175,-0.000750
50%,1.0,0 days 01:06:26.400000,99660.00000,0.529655,0.585035,0.066205,0.468540,0.514260,1.448702,0.479445,...,-0.000250,0.000000,0.000500,0.000000,-0.000250,0.000000,0.000000,-0.000750,-0.00025,0.000000
75%,1.0,0 days 01:39:39.580000,149489.50000,0.675230,0.804560,0.132057,0.605067,0.651310,2.573503,0.663420,...,0.009250,0.011500,0.010750,0.011500,0.010250,0.010750,0.011500,0.008500,0.01100,0.001000
max,1.0,0 days 02:12:52.760000,199319.00000,1.113850,1.488540,1.601429,0.966180,1.071450,11.293826,1.043250,...,0.441000,0.284000,0.260500,0.098750,0.138750,0.260750,0.087500,0.126750,0.44550,0.146750



=== 4. ESTADÍSTICAS DE VARIABLES CATEGÓRICAS Y EVENTOS ===


,set_pieces,players_in_the_box,players_in_the_box_team,ball_in_final_third,ball_in_final_third_team,ball_in_the_box,ball_in_the_box_team,goals,goals_team,progression,...,defending_in_middle_third,defending_in_middle_third_team,defensive_transition,defensive_transition_team,defending_in_attacking_third,defending_in_attacking_third_team,defending_in_defensive_third,defending_in_defensive_third_team,44_x_vel,44_y_vel
count,44037,81389,81389,61343,61343,29755,29755,16507,16507,31189,...,31189,31189,9134,9134,11033,11033,22413,22413,0,0
unique,6,7,2,1,2,1,2,1,2,1,...,1,2,1,2,1,2,1,3,0,0
top,Throw-in,1,Barcelona,True,Barcelona,True,Barcelona,True,Barcelona,True,...,True,Atletic Club,True,Atletic Club,True,Atletic Club,True,Atletic Club,NaN,NaN
freq,21000,14947,46197,61343,36165,29755,16718,16507,9979,31189,...,31189,23119,9134,5031,11033,7814,22413,14269,NaN,NaN


## 4. Save df

In [13]:
import os
import pandas as pd

# 1. Definir la ruta de la carpeta y el nombre del archivo
carpeta_destino = 'data'
nombre_archivo = 'dataframe_hackaton_final.csv'

# 2. Crear la carpeta 'data' si no existe
os.makedirs(carpeta_destino, exist_ok=True)

# 3. Construir la ruta completa del archivo
ruta_completa = os.path.join(carpeta_destino, nombre_archivo)

# 4. Guardar el DataFrame en CSV
# index=False evita que Pandas guarde el índice numérico como una columna extra
df_hackathon.to_csv(ruta_completa, index=False)

print(f"¡DataFrame guardado con éxito en: {ruta_completa}")

¡DataFrame guardado con éxito en: data\dataframe_hackaton_final.csv
